# Synthetic dynamics stress test

This notebook compares matched held-out manuscript-profile synthetic jobs for `aligned_dynamics` and `discordant_dynamics` using the same evaluation seed.

These are controlled sanity checks: velocity truth is generated by the frozen simulator to agree with displacement in `aligned_dynamics` and to oppose displacement in `discordant_dynamics`. The notebook does not claim biological validation. It reads frozen audit outputs and deterministically regenerates only the selected synthetic cell geometry and velocity arrays from the frozen job configs for visualization.

No ScGeo methods, thresholds, simulation truth definitions, frozen protocol settings, or frozen benchmark outputs are modified.

## Reader guide

- **Purpose:** Synthetic dynamics stress test in the frozen revision workflow.
- **Inference scope:** Controlled synthetic evaluation. One frozen simulation job/seed is the independent unit; no biological claim is made.
- **Inputs:** Checksum-pinned manuscript-profile benchmark tables selected by `SCGEO_BENCHMARK_DIR` and the frozen benchmark config.
- **Implementation:** `scripts/execute_revision_notebooks.py`.
- **Outputs:** Ignored `results/revision_synthetic_benchmark/` review artifacts.
- **Frozen findings:** Report accepted and negative controlled findings, including estimator, representation, local-geometry, dynamics, seed-dependence, recall, and coverage results, without changing thresholds.
- **Limitations:** Synthetic behavior is scenario-specific; controlled corruption sensitivity is not general OOD detection.

This tracked source notebook is intentionally output-free. The clean-kernel runner writes executed review copies and generated artifacts under the ignored results directory.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "scgeo_revision_matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.16,
})

STATE_COLORS = {
    "state_0": "#4C78A8",
    "state_1": "#F58518",
    "state_2": "#54A24B",
    "state_3": "#B279A2",
    "state_4": "#E45756",
}
CONDITION_MARKERS = {"control": "o", "treated": "^"}
EQUIVALENT_REPS = ["X_truth", "X_rotated", "X_scaled", "X_padded"]
SELECTED_SEED = 5
SELECTED_JOBS = [
    {"scenario": "aligned_dynamics", "job_id": "manuscript_aligned_dynamics_evaluation_seed5", "velocity_mode": "aligned", "title": "Aligned dynamics sanity check"},
    {"scenario": "discordant_dynamics", "job_id": "manuscript_discordant_dynamics_evaluation_seed5", "velocity_mode": "discordant", "title": "Discordant dynamics sanity check"},
]


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "configs" / "manuscript_benchmark_v1.json").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory")


REPO_ROOT = find_repo_root()
CONFIG = json.loads((REPO_ROOT / "configs" / "manuscript_benchmark_v1.json").read_text(encoding="utf-8"))


def resolve_config_path(value: str) -> Path:
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()


BENCHMARK_DIR = resolve_config_path(os.environ.get(CONFIG["benchmark_dir_env"], CONFIG["default_benchmark_dir"]))
SOURCE_REPO = resolve_config_path(os.environ.get(CONFIG["source_repository_env"], CONFIG["default_source_repository"]))
OUTPUT_DIR = resolve_config_path(os.environ.get(CONFIG["notebook_output_env"], CONFIG["default_output_dir"]))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def rel_display(path: Path) -> str:
    path = path.resolve()
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not-installed"


def git_commit(path: Path) -> str | None:
    if not (path / ".git").exists():
        return None
    try:
        return subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_selected_frozen_files(required_paths: list[str]) -> pd.DataFrame:
    manifest = pd.read_csv(REPO_ROOT / CONFIG["results_manifest"])
    manifest_idx = manifest.set_index("relative_path", drop=False)
    if set(manifest["protocol_version"].unique()) != {CONFIG["protocol_version"]}:
        raise AssertionError("Manifest protocol does not match config")
    if set(manifest["source_commit"].unique()) != {CONFIG["expected_source_commit"]}:
        raise AssertionError("Manifest source commit does not match config")
    missing = sorted(set(required_paths).difference(manifest_idx.index))
    if missing:
        raise AssertionError(f"Required selected-job files are absent from manifest: {missing}")
    rows = []
    for rel in required_paths:
        path = BENCHMARK_DIR / rel
        if not path.exists():
            raise FileNotFoundError(path)
        observed = sha256_file(path)
        expected = manifest_idx.loc[rel, "sha256"]
        if observed != expected:
            raise AssertionError(f"Checksum mismatch for {rel}")
        rows.append({"relative_path": rel, "sha256": observed, "size_bytes": int(path.stat().st_size)})
    source_commit = git_commit(SOURCE_REPO)
    if source_commit is not None and source_commit != CONFIG["expected_source_commit"]:
        raise AssertionError(f"Source repo commit mismatch: {source_commit}")
    return pd.DataFrame(rows)


def job_file(job_id: str, suffix: str) -> str:
    return f"{job_id}_{suffix}"


def read_job_csv(job_id: str, suffix: str) -> pd.DataFrame:
    return pd.read_csv(BENCHMARK_DIR / job_file(job_id, suffix))


def load_simulation_module():
    module_path = SOURCE_REPO / "scgeo" / "bench" / "_simulation.py"
    spec = importlib.util.spec_from_file_location("scgeo_bench_simulation_frozen", module_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load simulation module from {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def write_figure_bundle(stem: str, fig: plt.Figure, source_table: pd.DataFrame, alt_text: str) -> pd.DataFrame:
    fig_dir = OUTPUT_DIR / "figures"
    source_dir = OUTPUT_DIR / "figure_sources"
    alt_dir = OUTPUT_DIR / "alt_text"
    for directory in (fig_dir, source_dir, alt_dir):
        directory.mkdir(parents=True, exist_ok=True)
    png_path = fig_dir / f"{stem}.png"
    svg_path = fig_dir / f"{stem}.svg"
    csv_path = source_dir / f"{stem}.csv"
    alt_path = alt_dir / f"{stem}.txt"
    source_table.to_csv(csv_path, index=False)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", metadata={"Date": None})
    fig.savefig(png_path, format="png", bbox_inches="tight", metadata={"Software": "matplotlib"})
    alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
    plt.close(fig)
    return pd.DataFrame([{ 
        "figure": stem,
        "png": rel_display(png_path),
        "svg": rel_display(svg_path),
        "source_csv": rel_display(csv_path),
        "alt_text": rel_display(alt_path),
    }])


def write_table_artifact(stem: str, table: pd.DataFrame, alt_text: str | None = None) -> pd.DataFrame:
    source_dir = OUTPUT_DIR / "figure_sources"
    source_dir.mkdir(parents=True, exist_ok=True)
    csv_path = source_dir / f"{stem}.csv"
    table.to_csv(csv_path, index=False)
    row = {"artifact": stem, "source_csv": rel_display(csv_path)}
    if alt_text is not None:
        alt_dir = OUTPUT_DIR / "alt_text"
        alt_dir.mkdir(parents=True, exist_ok=True)
        alt_path = alt_dir / f"{stem}.txt"
        alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
        row["alt_text"] = rel_display(alt_path)
    return pd.DataFrame([row])


def write_metadata(stem: str, extra: dict[str, object] | None = None) -> pd.DataFrame:
    metadata_dir = OUTPUT_DIR / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    metadata = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "packages": {name: package_version(name) for name in ["scgeo", "anndata", "pandas", "numpy", "matplotlib", "nbformat", "nbclient"]},
        "notebook_repository_commit": git_commit(REPO_ROOT),
        "source_commit_expected": CONFIG["expected_source_commit"],
        "source_repository_commit": git_commit(SOURCE_REPO),
        "protocol_version": CONFIG["protocol_version"],
        "profile": CONFIG["profile"],
        "benchmark_dir": str(BENCHMARK_DIR),
        "selected_jobs": [item["job_id"] for item in SELECTED_JOBS],
        "selected_seed": SELECTED_SEED,
        "deterministic_regeneration": "simulate selected synthetic jobs from frozen job configs only; no ScGeo methods or thresholds are changed",
        "threshold_policy": CONFIG["threshold_policy"],
        "biological_validation_claimed": False,
        "dataset_d_plan_executed": False,
    }
    if extra:
        metadata.update(extra)
    path = metadata_dir / f"{stem}_metadata.json"
    path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return pd.DataFrame([{"metadata": rel_display(path)}])

In [ ]:
required_files = []
required_suffixes = [
    "config.json",
    "status.json",
    "runtime_summary.csv",
    "state_metrics.csv",
    "shift_estimate_diagnostics.csv",
    "representation_consensus.csv",
    "alignment_accuracy.csv",
    "bootstrap_coverage.csv",
    "coverage_summary.csv",
    "summary_metrics.csv",
    "representation_quality_outlier.csv",
    "neighborhood_discrimination.csv",
    "sample_center_diagnostics.csv",
]
for job in SELECTED_JOBS:
    required_files.extend(job_file(job["job_id"], suffix) for suffix in required_suffixes)
verified_files = verify_selected_frozen_files(required_files)

job_configs = {}
for job in SELECTED_JOBS:
    config = json.loads((BENCHMARK_DIR / job_file(job["job_id"], "config.json")).read_text(encoding="utf-8"))
    status = json.loads((BENCHMARK_DIR / job_file(job["job_id"], "status.json")).read_text(encoding="utf-8"))
    if config["seed_split"] != "evaluation":
        raise AssertionError(f"{job['job_id']} is not held-out evaluation")
    if int(config["seed"]) != SELECTED_SEED:
        raise AssertionError(f"{job['job_id']} does not use selected seed {SELECTED_SEED}")
    if config["scenario"] != job["scenario"]:
        raise AssertionError(f"Scenario mismatch for {job['job_id']}")
    if config.get("velocity_mode") != job["velocity_mode"]:
        raise AssertionError(f"Velocity mode mismatch for {job['job_id']}")
    if status.get("status") != "completed":
        raise AssertionError(f"{job['job_id']} is not completed")
    job_configs[job["job_id"]] = config

pd.DataFrame([{
    "selected_seed": SELECTED_SEED,
    "jobs": ",".join(job["job_id"] for job in SELECTED_JOBS),
    "protocol": CONFIG["protocol_version"],
    "source_commit": CONFIG["expected_source_commit"],
    "verified_files": len(verified_files),
    "controlled_sanity_check": True,
    "biological_validation_claimed": False,
}])

## Deterministic geometry and velocity regeneration

The frozen job CSVs contain state-level and representation-level audit outputs, not per-cell coordinates. The cells and velocity vectors below are regenerated deterministically from the selected frozen job configs, and cell counts are checked against frozen `sample_center_diagnostics` before plotting.

In [ ]:
sim = load_simulation_module()
adata_by_job = {}
for job in SELECTED_JOBS:
    cfg = job_configs[job["job_id"]]
    profile_cfg = cfg["profile_cfg"]
    adata = sim.simulate_perturbation_geometry(
        scenario=cfg["scenario"],
        n_samples_per_condition=int(profile_cfg["n_samples_per_condition"]),
        cells_per_sample=int(profile_cfg["cells_per_sample"]),
        seed=int(cfg["seed"]),
        velocity_mode=cfg.get("velocity_mode"),
    )
    runtime = read_job_csv(job["job_id"], "runtime_summary.csv")
    if int(runtime["n_obs"].iloc[0]) != int(adata.n_obs):
        raise AssertionError(f"Regenerated n_obs mismatch for {job['job_id']}")
    frozen_counts = read_job_csv(job["job_id"], "sample_center_diagnostics.csv")[["sample", "state", "n_cells"]].sort_values(["sample", "state"]).reset_index(drop=True)
    regen_counts = (
        adata.obs.groupby(["sample", "state"], observed=False)
        .size()
        .rename("n_cells")
        .reset_index()
        .sort_values(["sample", "state"])
        .reset_index(drop=True)
    )
    pd.testing.assert_frame_equal(frozen_counts, regen_counts, check_dtype=False)
    truth = adata.uns["simulation_truth"]
    if truth.get("velocity_keys") is None:
        raise AssertionError(f"Velocity keys are missing for {job['job_id']}")
    adata_by_job[job["job_id"]] = adata

pd.DataFrame([
    {
        "job_id": job["job_id"],
        "scenario": job["scenario"],
        "cells": int(adata_by_job[job["job_id"]].n_obs),
        "representations": ",".join(sorted(adata_by_job[job["job_id"]].obsm.keys())),
        "velocity_keys": ",".join(sorted(k for k in adata_by_job[job["job_id"]].uns["simulation_truth"]["velocity_keys"].values() if k is not None)),
        "count_check": "matches frozen sample_center_diagnostics",
    }
    for job in SELECTED_JOBS
])

In [ ]:
visual_rng = np.random.default_rng(20260717)
plot_rows = []
center_rows = []
velocity_rows = []

for job in SELECTED_JOBS:
    job_id = job["job_id"]
    adata = adata_by_job[job_id]
    obs = adata.obs.reset_index(names="cell_id")
    coords = np.asarray(adata.obsm["X_truth"])[:, :2]
    velocity = np.asarray(adata.obsm["V_truth"])[:, :2]
    selected_indices = []
    for (_, _), group in obs.groupby(["state", "condition"], observed=False):
        idx = group.index.to_numpy()
        take = min(160, len(idx))
        selected_indices.extend(visual_rng.choice(idx, size=take, replace=False).tolist())
    selected_indices = sorted(selected_indices)
    visual_obs = obs.loc[selected_indices].copy()
    for row_idx, obs_row in visual_obs.iterrows():
        x, y = coords[row_idx]
        plot_rows.append({
            "row_type": "cell_visual_subset",
            "job_id": job_id,
            "scenario": job["scenario"],
            "cell_id": obs_row["cell_id"],
            "state": obs_row["state"],
            "condition": obs_row["condition"],
            "x": float(x),
            "y": float(y),
        })
    coord_df = pd.DataFrame(coords, columns=["x", "y"])
    coord_df["vx"] = velocity[:, 0]
    coord_df["vy"] = velocity[:, 1]
    coord_df["state"] = adata.obs["state"].to_numpy()
    coord_df["condition"] = adata.obs["condition"].to_numpy()
    centers = coord_df.groupby(["state", "condition"], observed=False, as_index=False).agg(x=("x", "mean"), y=("y", "mean"))
    for _, center_row in centers.iterrows():
        center_rows.append({
            "row_type": "condition_center_all_cells",
            "job_id": job_id,
            "scenario": job["scenario"],
            "cell_id": "",
            "state": center_row["state"],
            "condition": center_row["condition"],
            "x": float(center_row["x"]),
            "y": float(center_row["y"]),
        })
    state_velocity = coord_df.groupby("state", observed=False, as_index=False).agg(vx=("vx", "mean"), vy=("vy", "mean"))
    midpoint = centers.pivot(index="state", columns="condition", values=["x", "y"])
    for _, vel_row in state_velocity.iterrows():
        state = vel_row["state"]
        if {"control", "treated"}.issubset(midpoint.loc[state].dropna().index.get_level_values(1)):
            x0 = float(midpoint.loc[state, ("x", "control")])
            x1 = float(midpoint.loc[state, ("x", "treated")])
            y0 = float(midpoint.loc[state, ("y", "control")])
            y1 = float(midpoint.loc[state, ("y", "treated")])
            velocity_rows.append({
                "row_type": "state_velocity_direction_all_cells",
                "job_id": job_id,
                "scenario": job["scenario"],
                "state": state,
                "condition": "all_cells",
                "x": (x0 + x1) / 2.0,
                "y": (y0 + y1) / 2.0,
                "dx": x1 - x0,
                "dy": y1 - y0,
                "vx": float(vel_row["vx"]),
                "vy": float(vel_row["vy"]),
            })

plot_source = pd.concat([pd.DataFrame(plot_rows), pd.DataFrame(center_rows)], ignore_index=True)
velocity_source = pd.DataFrame(velocity_rows)
plot_source.head()

In [ ]:
consensus_frames = []
alignment_frames = []
state_metric_frames = []
shift_diag_frames = []
bootstrap_frames = []
neighborhood_frames = []
summary_metric_frames = []
quality_frames = []
for job in SELECTED_JOBS:
    job_id = job["job_id"]
    consensus_frames.append(read_job_csv(job_id, "representation_consensus.csv").assign(job_id=job_id, scenario=job["scenario"]))
    alignment_frames.append(read_job_csv(job_id, "alignment_accuracy.csv").assign(job_id=job_id, scenario=job["scenario"]))
    state_metric_frames.append(read_job_csv(job_id, "state_metrics.csv").assign(job_id=job_id, scenario=job["scenario"]))
    shift_diag_frames.append(read_job_csv(job_id, "shift_estimate_diagnostics.csv").assign(job_id=job_id, scenario=job["scenario"]))
    bootstrap_frames.append(read_job_csv(job_id, "bootstrap_coverage.csv").assign(job_id=job_id, scenario=job["scenario"]))
    neighborhood_frames.append(read_job_csv(job_id, "neighborhood_discrimination.csv").assign(job_id=job_id, scenario=job["scenario"], job_label=job["title"]))
    summary_metric_frames.append(read_job_csv(job_id, "summary_metrics.csv").assign(job_id=job_id, scenario=job["scenario"]))
    quality_frames.append(read_job_csv(job_id, "representation_quality_outlier.csv").assign(job_id=job_id, scenario=job["scenario"]))

consensus = pd.concat(consensus_frames, ignore_index=True)
alignment = pd.concat(alignment_frames, ignore_index=True)
state_metrics = pd.concat(state_metric_frames, ignore_index=True)
shift_diag = pd.concat(shift_diag_frames, ignore_index=True)
bootstrap = pd.concat(bootstrap_frames, ignore_index=True)
neighborhood = pd.concat(neighborhood_frames, ignore_index=True)
summary_metrics = pd.concat(summary_metric_frames, ignore_index=True)
quality = pd.concat(quality_frames, ignore_index=True)

agreement_rows = []
for (job_id, scenario, state), group in alignment[(alignment["source"] == "single_representation") & alignment["rep"].isin(EQUIVALENT_REPS)].groupby(["job_id", "scenario", "state"], observed=False):
    consensus_row = alignment[(alignment["job_id"] == job_id) & (alignment["source"] == "consensus") & (alignment["state"] == state)].iloc[0]
    classes = group.set_index("rep").reindex(EQUIVALENT_REPS)["predicted_class"].to_dict()
    consensus_class = consensus_row["predicted_class"]
    agreement_rows.append({
        "job_id": job_id,
        "scenario": scenario,
        "state": state,
        "true_class": consensus_row["true_class"],
        "consensus_predicted_class": consensus_class,
        "consensus_correct": bool(consensus_row["correct"]),
        "equivalent_rep_classes": "; ".join(f"{rep}={classes.get(rep)}" for rep in EQUIVALENT_REPS),
        "equivalent_rep_consensus_agreement_fraction": float((group["predicted_class"] == consensus_class).mean()),
        "single_rep_all_correct_fraction": float(group["correct"].mean()),
        "velocity_requested": bool(consensus_row["velocity_requested"]),
    })
agreement_table = pd.DataFrame(agreement_rows).sort_values(["scenario", "state"])

neighbor_equiv = neighborhood[
    (neighborhood["rep_a"] == "X_truth")
    & (neighborhood["rep_b"].isin(["X_rotated", "X_scaled", "X_padded"]))
    & (neighborhood["k"] == 30)
]
neighbor_summary = (
    neighbor_equiv.pivot_table(index=["job_id", "scenario", "rep_b", "pair_class"], columns="metric", values="value", aggfunc="first")
    .reset_index()
    .sort_values(["scenario", "rep_b"])
)

write_table_artifact("05_equivalent_representation_agreement", agreement_table, "Equivalent-representation dynamics agreement table for matched aligned and discordant synthetic held-out jobs. Shifted states agree across all equivalent representations; neutral state_2 shows discordant single-representation velocity classes while consensus remains stable_neutral, keeping negative/caveat evidence visible.")
write_table_artifact("05_equivalent_neighborhood_summary", neighbor_summary)
agreement_table

In [ ]:
# Merge frozen consensus cosine with regenerated first-two-dimension velocity arrows for plotting.
velocity_plot = velocity_source.merge(
    consensus[["job_id", "scenario", "state", "consensus_label", "alignment_cosine_median", "aligned_fraction", "discordant_fraction", "neutral_fraction"]],
    on=["job_id", "scenario", "state"],
    how="left",
)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8), constrained_layout=True)
for ax, job in zip(axes, SELECTED_JOBS):
    job_id = job["job_id"]
    rep_cells = plot_source[(plot_source["job_id"] == job_id) & (plot_source["row_type"] == "cell_visual_subset")]
    rep_centers = plot_source[(plot_source["job_id"] == job_id) & (plot_source["row_type"] == "condition_center_all_cells")]
    rep_velocity = velocity_plot[velocity_plot["job_id"] == job_id]
    for condition, marker in CONDITION_MARKERS.items():
        subset = rep_cells[rep_cells["condition"] == condition]
        for state, state_subset in subset.groupby("state", observed=False):
            ax.scatter(
                state_subset["x"],
                state_subset["y"],
                s=9,
                alpha=0.25 if condition == "control" else 0.38,
                marker=marker,
                color=STATE_COLORS.get(state, "#777777"),
                linewidths=0,
            )
    max_v = max(1e-8, float(np.sqrt(rep_velocity["vx"] ** 2 + rep_velocity["vy"] ** 2).max()))
    velocity_scale = 0.72 / max_v
    for state, state_centers in rep_centers.groupby("state", observed=False):
        wide = state_centers.set_index("condition")
        if {"control", "treated"}.issubset(wide.index):
            x0, y0 = wide.loc["control", ["x", "y"]]
            x1, y1 = wide.loc["treated", ["x", "y"]]
            color = STATE_COLORS.get(state, "#777777")
            ax.scatter([x0], [y0], marker="o", s=42, color=color, edgecolor="black", linewidth=0.5, zorder=4)
            ax.scatter([x1], [y1], marker="^", s=50, color=color, edgecolor="black", linewidth=0.5, zorder=4)
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops={"arrowstyle": "->", "color": color, "lw": 1.3, "alpha": 0.95})
            vrow = rep_velocity[rep_velocity["state"] == state].iloc[0]
            vx = float(vrow["vx"]) * velocity_scale
            vy = float(vrow["vy"]) * velocity_scale
            ax.annotate("", xy=(float(vrow["x"]) + vx, float(vrow["y"]) + vy), xytext=(float(vrow["x"]), float(vrow["y"])), arrowprops={"arrowstyle": "-|>", "color": "#222222", "lw": 1.25, "linestyle": "--", "alpha": 0.86})
            ax.text(x1, y1, f"{state.replace('state_', 's')}\ncos={vrow['alignment_cosine_median']:.2f}\n{vrow['consensus_label'].replace('stable_', '')}", fontsize=7.2, color="black", ha="left", va="bottom")
    consensus_counts = consensus.loc[consensus["job_id"] == job_id, "consensus_label"].value_counts().to_dict()
    summary_row = summary_metrics[(summary_metrics["job_id"] == job_id) & (summary_metrics["metric"] == "alignment_class_accuracy")].iloc[0]
    panel_note = (
        f"controlled truth: {job['velocity_mode']}\n"
        f"consensus counts: {consensus_counts}\n"
        f"alignment accuracy table value: {summary_row['value']:.2f}\n"
        "black dashed arrows: state-level velocity direction"
    )
    ax.text(0.02, 0.98, panel_note, transform=ax.transAxes, ha="left", va="top", fontsize=8, bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#cccccc", "alpha": 0.88})
    ax.set_title(f"{job['title']}\n{job_id}")
    ax.set_xlabel("X_truth dimension 1")
    ax.set_ylabel("X_truth dimension 2")

state_handles = [plt.Line2D([0], [0], marker="o", color="none", markerfacecolor=color, label=state, markersize=6) for state, color in STATE_COLORS.items()]
axes[0].legend(handles=state_handles, title="State", loc="lower left", frameon=True, fontsize=8)
arrow_handles = [
    plt.Line2D([0], [0], color="#555555", lw=1.4, label="condition displacement"),
    plt.Line2D([0], [0], color="#222222", lw=1.4, linestyle="--", label="velocity direction"),
]
axes[1].legend(handles=arrow_handles, title="Arrows", loc="lower left", frameon=True, fontsize=8)

figure_source = pd.concat([
    plot_source.assign(source_table="cells_and_centers"),
    velocity_plot.assign(source_table="state_velocity_and_frozen_consensus", cell_id="", condition="all_cells"),
], ignore_index=True, sort=False)
alt = (
    "Synthetic dynamics stress-test figure comparing matched held-out seed 5 for aligned and discordant controlled scenarios. "
    "Cells are colored by state and shaped by condition; colored arrows show control-to-treated displacement; black dashed arrows show regenerated state-level velocity direction. "
    "Frozen cosine labels are positive for stable_aligned shifted states and negative for stable_discordant shifted states. Neutral-state caveats remain visible in the source tables. "
    "These panels are controlled synthetic sanity checks, not biological validation."
)
figure_artifacts = write_figure_bundle("05_synthetic_dynamics_stress_test", fig, figure_source, alt)
figure_artifacts

In [ ]:
robust = state_metrics[state_metrics["method"] == "robust_geometric_median"].copy()
consensus_keep = consensus[[
    "job_id", "scenario", "state", "consensus_label", "status", "n_usable_representations", "usable_fraction",
    "alignment_cosine_median", "aligned_fraction", "discordant_fraction", "neutral_fraction", "velocity_requested",
    "loo_rep_magnitude_max_relative_deviation",
]].copy()
bootstrap_keep = bootstrap[["job_id", "scenario", "state", "coverage_applicable", "coverage_status", "covered"]].copy()
shift_keep = shift_diag[["job_id", "scenario", "state", "delta_norm", "normalized_delta_norm", "bootstrap_ci95_low", "bootstrap_ci95_high", "direction_stability", "sign_stability"]].copy()

evidence = (
    robust[["job_id", "scenario", "state", "estimated_magnitude", "true_magnitude", "predicted_shifted", "true_shifted", "threshold"]]
    .merge(shift_keep, on=["job_id", "scenario", "state"], how="left")
    .merge(consensus_keep, on=["job_id", "scenario", "state"], how="left")
    .merge(agreement_table, on=["job_id", "scenario", "state"], how="left")
    .merge(bootstrap_keep, on=["job_id", "scenario", "state"], how="left")
)

def robust_label(row):
    return (
        f"pred={bool(row['predicted_shifted'])}; true={bool(row['true_shifted'])}; "
        f"delta={row['delta_norm']:.3f}; CI=[{row['bootstrap_ci95_low']:.3f},{row['bootstrap_ci95_high']:.3f}]"
    )

def coverage_label(row):
    if bool(row["coverage_applicable"]):
        return f"bootstrap {row['coverage_status']}; covered={row['covered']}"
    return f"bootstrap {row['coverage_status']}; zero-effect state not a magnitude-coverage target"

def caveat_label(row):
    notes = []
    if row["consensus_label"] == "stable_neutral" and row["equivalent_rep_consensus_agreement_fraction"] < 1.0:
        notes.append("single-representation velocity classes disagree with neutral consensus")
    if row["consensus_label"] in {"stable_aligned", "stable_discordant"} and abs(float(row["alignment_cosine_median"])) < 0.9:
        notes.append("weaker velocity-displacement cosine")
    if not notes:
        notes.append("no unavailable evidence for this controlled synthetic state")
    return "; ".join(notes)

evidence["robust_effect"] = evidence.apply(robust_label, axis=1)
evidence["dynamics_status"] = evidence.apply(lambda row: f"{row['consensus_label']} cosine={row['alignment_cosine_median']:.3f}; aligned_frac={row['aligned_fraction']:.2f}; discordant_frac={row['discordant_fraction']:.2f}; neutral_frac={row['neutral_fraction']:.2f}", axis=1)
evidence["equivalent_representation_agreement"] = evidence.apply(lambda row: f"agreement={row['equivalent_rep_consensus_agreement_fraction']:.2f}; classes: {row['equivalent_rep_classes']}", axis=1)
evidence["coverage_status_reason"] = evidence.apply(coverage_label, axis=1)
evidence["negative_or_unavailable_evidence"] = evidence.apply(caveat_label, axis=1)

evidence_table = evidence[[
    "scenario",
    "state",
    "robust_effect",
    "dynamics_status",
    "equivalent_representation_agreement",
    "coverage_status_reason",
    "negative_or_unavailable_evidence",
    "estimated_magnitude",
    "true_magnitude",
    "threshold",
    "alignment_cosine_median",
    "consensus_label",
    "true_class",
    "consensus_predicted_class",
    "consensus_correct",
    "single_rep_all_correct_fraction",
]].sort_values(["scenario", "state"])

evidence_alt = (
    "State-level evidence table for matched aligned and discordant controlled synthetic dynamics jobs. "
    "The table lists robust displacement effect, frozen displacement-velocity cosine and consensus class, agreement across equivalent representations, bootstrap coverage reason, and caveats. "
    "Neutral state_2 keeps negative evidence visible because its single-representation velocity classes are discordant even when consensus remains stable_neutral."
)
write_table_artifact("05_state_evidence_table", evidence_table, evidence_alt)
evidence_table

In [ ]:
negative_summary = pd.DataFrame([
    {
        "item": "controlled sanity-check scope",
        "evidence": "velocity truth is generated by the synthetic simulator to agree with displacement in aligned_dynamics and oppose displacement in discordant_dynamics; this is not biological validation",
    },
    {
        "item": "neutral-state single-representation caveat",
        "evidence": "state_2 is stable_neutral by consensus in both jobs, but equivalent single-representation velocity classes are discordant in the frozen alignment table",
    },
    {
        "item": "local-distortion layer unavailable as a positive target",
        "evidence": "summary_metrics representation_local_distortion_detection has zero TP/FP/FN in aligned and discordant dynamics because these scenarios are dynamics sanity checks, not local-distortion truth jobs",
    },
    {
        "item": "zero-effect coverage status",
        "evidence": "states 2-4 are zero-effect states and are marked not_applicable for magnitude coverage rather than counted as coverage failures",
    },
])
write_table_artifact("05_negative_and_unavailable_evidence", negative_summary)
negative_summary

## Public pancreatic-development workflow plan

This section is a prespecified plan only. It is not executed in this notebook. The goal would be to test the same displacement/dynamics reporting on a public scVelo/CellRank endocrine pancreas dataset, without retuning ScGeo thresholds after seeing the result.

In [ ]:
dataset_d_plan = pd.DataFrame([
    {
        "stage": "data acquisition",
        "planned_actions": "Use the public endocrine pancreas dataset distributed through scVelo/CellRank examples; record URL or loader, download date, raw/checkpoint checksums, organism, batch/sample annotations, and preprocessing state.",
        "outputs": "data provenance manifest; immutable raw and processed AnnData checksums; exclusion log for missing annotations",
        "not_executed_here": True,
    },
    {
        "stage": "scVelo dynamical velocity",
        "planned_actions": "Run standard scVelo preprocessing, moments, recover_dynamics, and velocity in dynamical mode with pinned package versions and prespecified genes/cell filters from public tutorial defaults or documented preprocessing.",
        "outputs": "velocity graph, latent time, per-gene dynamical fit diagnostics, velocity confidence summaries",
        "not_executed_here": True,
    },
    {
        "stage": "CellRank kernels and terminal states",
        "planned_actions": "Build a CellRank VelocityKernel from the scVelo dynamical velocities, combine with connectivity kernel only if prespecified, estimate macrostates/terminal states, and record fate probabilities.",
        "outputs": "terminal-state assignments, fate probability table, kernel parameters, transition matrix summaries",
        "not_executed_here": True,
    },
    {
        "stage": "prespecified representation ensemble",
        "planned_actions": "Define representations before outcome inspection: PCA, scVI or Harmony-integrated latent space if available, diffusion map, UMAP for display only, and a controlled rotated/scaled PCA sanity representation; document any representation excluded for missing data.",
        "outputs": "representation manifest, neighbor-preservation table, representation-quality diagnostics",
        "not_executed_here": True,
    },
    {
        "stage": "ScGeo displacement and dynamics analysis",
        "planned_actions": "Run ScGeo robust displacement by endocrine state or lineage stage, representation stability across the prespecified ensemble, local geometry stability, and velocity-displacement alignment using the scVelo velocity basis without changing frozen synthetic thresholds.",
        "outputs": "state-level displacement table, dynamics consensus labels, local-geometry table, coverage/status reasons",
        "not_executed_here": True,
    },
    {
        "stage": "expected figures",
        "planned_actions": "Prepare matched panels for cells by state/condition or pseudotime bin, displacement arrows, velocity arrows, CellRank fate arrows, cosine heatmap, representation-stability heatmap, local-distortion map, and per-state evidence table.",
        "outputs": "PNG/SVG figures, source CSVs, deterministic alt text, metadata JSON",
        "not_executed_here": True,
    },
    {
        "stage": "comparator outputs",
        "planned_actions": "Compare ScGeo dynamics labels against scVelo latent-time direction, CellRank terminal-state/fate-probability direction, simple centroid shifts, and single-representation velocity-displacement cosine; report disagreements rather than retuning thresholds.",
        "outputs": "comparator summary table, discordance table, sensitivity limited to prespecified display summaries",
        "not_executed_here": True,
    },
])
plan_alt = "Dataset D plan for the public scVelo/CellRank endocrine pancreas dataset. The plan covers acquisition, scVelo dynamical velocity, CellRank VelocityKernel and terminal states, a prespecified representation ensemble, ScGeo displacement/dynamics analysis, expected figures, and comparator outputs. It is prepared but not executed."
write_table_artifact("05_dataset_d_public_pancreas_plan", dataset_d_plan, plan_alt)
dataset_d_plan

In [ ]:
metadata = write_metadata("05_synthetic_dynamics_stress_test", {
    "figures": ["05_synthetic_dynamics_stress_test"],
    "verified_frozen_files": int(len(verified_files)),
    "regenerated_jobs": [job["job_id"] for job in SELECTED_JOBS],
    "regenerated_cells_per_job": {job["job_id"]: int(adata_by_job[job["job_id"]].n_obs) for job in SELECTED_JOBS},
    "visual_subset_cells_per_state_condition_max": 160,
    "new_thresholds_tuned": False,
    "scgeo_methods_changed": False,
    "controlled_sanity_check_statement": "velocity truth is generated to agree with or oppose displacement; no biological validation is claimed",
    "dataset_d_plan_executed": False,
})
metadata